# nb_preparar_ciclo
Primeiro passo do pipeline mestre. Lê `ctl_tabelas` e devolve, por origem, a lista de tabelas do ciclo
com as consultas prontas:
- `query_max`: lida pelo Lookup na origem (novo watermark, lido **antes** da cópia)
- `query_copy`: consulta da Copy activity, com o marcador `#NOVO#` trocado pelo pipeline

Ciclo `AUTO`: antes das 09:00 (Brasília) = COMPLETO (dimensões + fatos); depois = FATOS.

In [ ]:
ciclo = "AUTO"        # AUTO | COMPLETO | FATOS
lote = "manual"       # o pipeline envia @pipeline().RunId
origens = ""          # filtro opcional, ex.: "NEXTI" ou "GS,4S" (vazio = todas)
hora_corte_completo = 9

In [ ]:
import os, json, datetime as dt
import pyarrow as pa
import duckdb
from deltalake import DeltaTable, write_deltalake

LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def _opts():
        return None
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
else:
    import notebookutils
    _WS = notebookutils.runtime.context["currentWorkspaceName"]
    def _opts():
        return {"bearer_token": notebookutils.credentials.getToken("storage"),
                "use_fabric_endpoint": "true"}
    def caminho(lakehouse, tabela):
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)

def existe(lakehouse, tabela):
    try:
        DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
        return True
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None):
    dt_ = DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
    return dt_.to_pyarrow_table(filters=filtros) if filtros else dt_.to_pyarrow_table()

def gravar(lakehouse, tabela, dados, modo="append"):
    kw = {"schema_mode": "overwrite"} if modo == "overwrite" else {}
    write_deltalake(caminho(lakehouse, tabela), dados, mode=modo,
                    storage_options=_opts(), **kw)

def tabela_delta(lakehouse, tabela):
    return DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())

def para_arrow(rel):
    """Converte resultado DuckDB em pyarrow.Table (compatível com várias versões)."""
    r = rel.fetch_arrow_table() if hasattr(rel, "fetch_arrow_table") else rel.arrow()
    return r.read_all() if hasattr(r, "read_all") else r

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

In [ ]:
# Alias "novo" entre aspas no Firebird para o Lookup receber a coluna em minúsculas.
MARCADOR = "#NOVO#"

SQL = {
    "SQLSERVER": {
        "agora": "SYSDATETIME()",
        "q_agora": "SELECT CONVERT(varchar(30), SYSDATETIME(), 126) AS novo",
        "q_max_data": "SELECT CONVERT(varchar(30), MAX({col}), 126) AS novo FROM {tab}",
        "q_max_id": "SELECT CAST(MAX({col}) AS varchar(30)) AS novo FROM {tab}",
        "fmt_data": "%Y-%m-%dT%H:%M:%S",
        "dias_atras": "DATEADD(DAY, -{dias}, CAST(GETDATE() AS date))",
    },
    "FIREBIRD": {
        "agora": "CURRENT_TIMESTAMP",
        "q_agora": 'SELECT CAST(CURRENT_TIMESTAMP AS VARCHAR(30)) AS "novo" FROM RDB$DATABASE',
        "q_max_data": 'SELECT CAST(MAX({col}) AS VARCHAR(30)) AS "novo" FROM {tab}',
        "q_max_id": 'SELECT CAST(MAX({col}) AS VARCHAR(30)) AS "novo" FROM {tab}',
        "fmt_data": "%Y-%m-%d %H:%M:%S",
        "dias_atras": "DATEADD(-{dias} DAY TO CURRENT_DATE)",
    },
}

def inicio_janela(ultimo, minutos, fmt):
    """Watermark anterior menos a janela de segurança, no formato literal do dialeto."""
    base = dt.datetime.strptime(ultimo.strip()[:19].replace(" ", "T"), "%Y-%m-%dT%H:%M:%S")
    return (base - dt.timedelta(minutes=minutos or 0)).strftime(fmt)

def montar(r, dialeto):
    s = SQL[dialeto]
    tab, col = r["tabela_origem"], r["coluna_watermark"]
    modo = r["modo_carga"]
    ultimo = (r["ultimo_watermark"] or "").strip()
    colunas = r["colunas"] or "*"
    if colunas.strip() == "*" and dialeto == "FIREBIRD":
        colunas = f"{tab}.*"   # Firebird não aceita '*' seguido de outras colunas
    tecnicas = f"'{r['origem']}' AS \"_origem\", '{lote}' AS \"_lote\", {s['agora']} AS \"_dt_carga\""
    filtros = []

    if modo == "INCREMENTAL" and r["tipo_watermark"] == "ID":
        q_max = s["q_max_id"].format(col=col, tab=tab)
        if ultimo:
            filtros.append(f"{col} > {int(ultimo)}")
        filtros.append(f"{col} <= {MARCADOR}")
    elif modo == "INCREMENTAL":
        q_max = s["q_max_data"].format(col=col, tab=tab)
        if ultimo:
            filtros.append(f"{col} > '{inicio_janela(ultimo, r['janela_seguranca_min'], s['fmt_data'])}'")
        filtros.append(f"{col} <= '{MARCADOR}'")
    elif modo == "JANELA":
        q_max = s["q_agora"]
        filtros.append(f"{col} >= {s['dias_atras'].format(dias=int(r['janela_dias']))}")
    else:  # FULL
        q_max = s["q_agora"]

    if r["filtro_adicional"]:
        filtros.append(f"({r['filtro_adicional']})")
    where = (" WHERE " + " AND ".join(filtros)) if filtros else ""
    return {
        "id": r["id"],
        "tabela_destino": nome_bronze(r["tabela_destino"]),   # destino da Copy activity
        "modo_carga": modo,
        "wm_atual": ultimo,
        "query_max": q_max,
        "query_copy": f"SELECT {colunas}, {tecnicas} FROM {tab}{where}",
    }

In [ ]:
if ciclo == "AUTO":
    ciclo = "COMPLETO" if dt.datetime.now(FUSO_BRT).hour < hora_corte_completo else "FATOS"
tipos = {"DIM", "FATO"} if ciclo == "COMPLETO" else {"FATO"}

filtro = {o.strip().upper() for o in origens.split(",") if o.strip()}
origens = {o["origem"]: o for o in ler(LH_BRONZE, "ctl_origens").to_pylist()}
saida = {nome: [] for nome in origens}          # toda origem aparece, mesmo vazia
for r in ler(LH_BRONZE, "ctl_tabelas").to_pylist():
    o = origens.get(r["origem"])
    # origens API são lidas pelo nb_ingestao_api, não pela cópia do pipeline
    if (r["ativo"] and o and o["ativo"] and o["dialeto"] != "API" and r["tipo"] in tipos
            and (not filtro or r["origem"] in filtro)):
        saida[r["origem"]].append(montar(r, o["dialeto"]))

resumo = {k: len(v) for k, v in saida.items()}
print(f"Ciclo {ciclo} | lote {lote} | tabelas por origem: {resumo}")
sair(json.dumps({"ciclo": ciclo, "lote": lote, "origens": saida}, ensure_ascii=False))